# Week 2 — Why the Guarantees Hold: entropy, unicity, and the one-time pad

**Lesson plan:** [`../weeks/week-02.md`](../weeks/week-02.md)
**Reading:** Shannon (1949), *Communication Theory of Secrecy Systems* — the
perfect-secrecy result (skim; it is dense).

Pure Python, no lab target. Week 1 *broke* a cipher; this week asks the deeper
question — **when is a cipher unbreakable, and what does that guarantee cost?** It
is the course's first look at a control with a *provable* guarantee (the one-time
pad), and at how trivially that guarantee is destroyed by misuse.

> ### The one idea
> Shannon turned "is this secure?" from opinion into arithmetic. **Entropy**
> measures the attacker's uncertainty; **unicity distance** says how much
> ciphertext collapses that uncertainty to one answer; and **perfect secrecy** is
> the single point where the ciphertext reveals *nothing* — achievable, provably,
> and almost never usable. Then we destroy it in three lines by reusing a key.

## 1 · Entropy — measuring what the attacker doesn't know

A key's strength isn't its length in characters; it's the **entropy** — the
attacker's uncertainty, in bits. For a key drawn uniformly from a space of size N,
that's log₂(N) bits. Compare a few schemes.

In [7]:
import math

schemes = [
    ("Caesar shift (26 keys)", 26),
    ("substitution (26! keys)", math.factorial(26)),
    ("56-bit DES key", 2**56),
    ("128-bit AES key", 2**128),
]
print(f"  {'scheme':<28}{'keyspace':>26}{'entropy (bits)':>16}")
print("  " + "-" * 70)
for name, N in schemes:
    print(f"  {name:<28}{N:>26.3g}{math.log2(N):>16.1f}")

print("""
Entropy, not key 'length', is the honest measure. The substitution cipher has 88
bits of key entropy — more than DES — yet week 1 broke it in under a second. So key
entropy is necessary but NOT sufficient. What went wrong? The PLAINTEXT leaked.""")

  scheme                                        keyspace  entropy (bits)
  ----------------------------------------------------------------------
  Caesar shift (26 keys)                              26             4.7
  substitution (26! keys)                       4.03e+26            88.4
  56-bit DES key                                7.21e+16            56.0
  128-bit AES key                                3.4e+38           128.0

Entropy, not key 'length', is the honest measure. The substitution cipher has 88
bits of key entropy — more than DES — yet week 1 broke it in under a second. So key
entropy is necessary but NOT sufficient. What went wrong? The PLAINTEXT leaked.


## 2 · Unicity distance — how much ciphertext betrays the key

Shannon's unicity distance answers: *how much ciphertext must an attacker see
before only one key is consistent with it?*

# U = H(K) / D

where H(K) is key entropy (bits) and **D is the redundancy of the language**
(bits/char that are predictable — English is ~3.2). Below U, many keys produce
plausible plaintext; above U, the real key is (in principle) pinned down.

In [2]:
H_K = math.log2(math.factorial(26))     # substitution key entropy
D = 3.2                                  # English redundancy, bits/char (Shannon)
U = H_K / D
print(f"substitution cipher: H(K) = {H_K:.1f} bits, redundancy D = {D} bits/char")
print(f"unicity distance U = H(K) / D = {U:.1f} characters\n")
print("Below ~28 characters: multiple keys give readable English — ambiguous.")
print("Above ~28 characters: the real key is uniquely determined in principle.")
print("""
This is EXACTLY why week 1 worked: the message was hundreds of characters, far past
U, so the key was pinned and hill-climbing found it. Unicity distance is the theory
behind that attack — and it says a short enough message could NOT have been broken
uniquely. Redundancy is the leak; compress the plaintext and U grows.""")

substitution cipher: H(K) = 88.4 bits, redundancy D = 3.2 bits/char
unicity distance U = H(K) / D = 27.6 characters

Below ~28 characters: multiple keys give readable English — ambiguous.
Above ~28 characters: the real key is uniquely determined in principle.

This is EXACTLY why week 1 worked: the message was hundreds of characters, far past
U, so the key was pinned and hill-climbing found it. Unicity distance is the theory
behind that attack — and it says a short enough message could NOT have been broken
uniquely. Redundancy is the leak; compress the plaintext and U grows.


## 3 · The one-time pad — perfect secrecy, provably

Make the key **as long as the message, truly random, and used once.** Encryption is
XOR. Shannon proved this achieves **perfect secrecy**: the ciphertext is
statistically independent of the plaintext — it reveals *nothing*, against an
adversary with unlimited computation.

The intuition, made concrete: for *any* guessed plaintext of the right length,
there exists a key making the ciphertext decrypt to it. So the ciphertext cannot
favor the true message over any other.

In [8]:
import os

def xor(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

msg = b"ATTACK AT DAWN"
key = os.urandom(len(msg))         # random, same length as the message, used ONCE
ct = xor(msg, key)
print("ciphertext:", ct.hex())
print("decrypts correctly:", xor(ct, key) == msg, "->", xor(ct, key).decode())

# Perfect secrecy: the SAME ciphertext decrypts to a totally different message
# under a different key. So an attacker seeing ct learns nothing about which.
fake = b"RETREAT NOW!!!"          # same length, opposite meaning
key_that_gives_fake = xor(ct, fake)
print("\nunder another key, the SAME ciphertext yields:",
      xor(ct, key_that_gives_fake).decode())
print("""
Both decryptions are equally 'valid' — there is a key for every possible plaintext.
The ciphertext cannot betray the real message because it is consistent with all of
them. THAT is perfect secrecy: a guarantee (axis 2) with no condition on the
attacker's compute. Unbreakable, and — as section 4 shows — almost unusable.""")

ciphertext: 7be005a9db738ab79d29acccb431
decrypts correctly: True -> ATTACK AT DAWN

under another key, the SAME ciphertext yields: RETREAT NOW!!!

Both decryptions are equally 'valid' — there is a key for every possible plaintext.
The ciphertext cannot betray the real message because it is consistent with all of
them. THAT is perfect secrecy: a guarantee (axis 2) with no condition on the
attacker's compute. Unbreakable, and — as section 4 shows — almost unusable.


## 4 · ⚠️ The catch, and the classic mistake: reuse the key once

Perfect secrecy requires the key be as long as the message, random, and **used
exactly once**. The first two are merely expensive. The third is the trap that has
sunk real systems (VENONA, MS-PPTP, the "two-time pad").

Reuse a key across two messages and the guarantee evaporates instantly, because
the key **cancels**:

# C₁ ⊕ C₂ = (P₁ ⊕ K) ⊕ (P₂ ⊕ K) = P₁ ⊕ P₂

The attacker never needed the key — they have the XOR of the two plaintexts, and
plaintext structure does the rest.

In [9]:
key = os.urandom(80)                # ONE key...
p1 = b"the launch code is four seven two the target is the north bridge tonight"
p2 = b"please water my plants and feed the cat while i am away for the weekend ok"
n = min(len(p1), len(p2)); p1, p2 = p1[:n], p2[:n]
c1, c2 = xor(p1, key), xor(p2, key)     # ...used TWICE

x = xor(c1, c2)
print("attacker computes c1 XOR c2, and the key is gone:")
print("  c1 XOR c2 == p1 XOR p2 ?", x == xor(p1, p2))

attacker computes c1 XOR c2, and the key is gone:
  c1 XOR c2 == p1 XOR p2 ? True


### Crib-dragging — turning P₁ ⊕ P₂ into the plaintexts

The attacker guesses a common word (a **crib**) appears somewhere in one message
and XORs it against `c1 ⊕ c2` at each position. Where the crib is correct, the
*other* message's text appears. Recognizable fragments amid noise = the crib's
location; the analyst walks cribs until both messages are reconstructed.

In [10]:
def printable_word(b):
    return all(c == 32 or 97 <= c <= 122 for c in b)   # lowercase + spaces only

for crib in (b"please", b"target", b" the "):
    print(f"crib {crib!r}:")
    for i in range(n - len(crib)):
        frag = xor(x[i:i + len(crib)], crib)
        if printable_word(frag):
            print(f"    pos {i:2d}: reveals {frag!r} in the OTHER message")
    print()

print("Dragging 'please' hits position 0 -> 'the la' (start of message 1);")
print("dragging 'target' hits a position in message 2. Real fragments surface at")
print("the true spots, noise elsewhere — the analyst recognizes the signal.")

crib b'please':
    pos  0: reveals b'the la' in the OTHER message
    pos 48: reveals b'ei  jk' in the OTHER message

crib b'target':
    pos 38: reveals b't whil' in the OTHER message
    pos 65: reveals b'ekwklx' in the OTHER message

crib b' the ':
    pos  6: reveals b'umjye' in the OTHER message
    pos 31: reveals b'wo th' in the OTHER message
    pos 33: reveals b'he ca' in the OTHER message
    pos 47: reveals b' am a' in the OTHER message
    pos 51: reveals b'amfnt' in the OTHER message
    pos 59: reveals b'ridge' in the OTHER message

Dragging 'please' hits position 0 -> 'the la' (start of message 1);
dragging 'target' hits a position in message 2. Real fragments surface at
the true spots, noise elsewhere — the analyst recognizes the signal.


In [6]:
# With a full crib (or once enough fragments are chained), recovery is total.
# If the attacker knows/guesses p1 entirely, the keystream falls out, and p2 with it.
keystream = xor(c1, p1)             # = key, recovered without ever guessing it
p2_recovered = xor(c2, keystream)
print("given p1 as a crib, p2 recovered in full:", p2_recovered == p2)
print(" ->", p2_recovered.decode())
print("""
'Perfect secrecy' used twice is no secrecy at all. The guarantee (axis 2) had a
CONDITION — use the key once — and violating the condition didn't weaken the cipher,
it collapsed it completely. Naming the condition is the whole skill (week 1).""")

given p1 as a crib, p2 recovered in full: True
 -> please water my plants and feed the cat while i am away for the weekend 

'Perfect secrecy' used twice is no secrecy at all. The guarantee (axis 2) had a
CONDITION — use the key once — and violating the condition didn't weaken the cipher,
it collapsed it completely. Naming the condition is the whole skill (week 1).


## 5 · The scorecard view — a guarantee and its price

| Property | One-time pad |
|---|---|
| **Guarantee (axis 2)** | perfect secrecy — ciphertext independent of plaintext, **against unbounded compute** |
| **Condition** | key random, ≥ message length, **used exactly once** |
| **Failure mode** | reuse → C₁⊕C₂ = P₁⊕P₂ → total break by crib-dragging |
| **Why unusable** | key distribution is as hard as message distribution (you must share as much secret key as you have plaintext) |

> The OTP is the *only* cipher with unconditional perfect secrecy — and it is
> almost never used, because its guarantee costs more than it's worth. Real crypto
> (weeks 3–4: AES, RSA) trades perfect secrecy for **computational** security:
> breakable in principle, infeasible in practice, and *usable*. The rest of the
> crypto unit is about that trade, and about the conditions those guarantees carry.

## 6 · Your studio deliverable

In `week02/`:

1. **Entropy & unicity** — compute H(K) and the unicity distance for two ciphers;
   explain, in terms of U, why week 1's attack succeeded and at what message length
   it would have become ambiguous.
2. **One-time pad** — implement it; demonstrate perfect secrecy by exhibiting two
   keys that decrypt one ciphertext to two different meaningful messages.
3. **Two-time pad break** — given two ciphertexts under a reused key, recover both
   plaintexts by crib-dragging. Report which cribs cracked which positions.
4. **Control Scorecard** — state the OTP's guarantee *and its condition* (axis 2),
   and the failure mode when the condition is violated. Then answer: why does
   nobody use it?

> Keep it rigorous — this is the week the course does not hand-wave Shannon — but it
> is deliberately **one** session of theory. The two-time-pad break is the payload:
> a provable guarantee, destroyed by one misuse.